# φ₁-χ Parameter Space Analysis for Molecular Interactions

Publication-quality multiplot visualization of binding energy surfaces, structural parameters, and zeta dihedral angle distributions in φ₁-χ parameter space for SS interactions.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
import os

interaction_type = 'DD'

# Publication-quality settings
plt.rcParams.update({
    # 'font.family': 'Arial', 
    'font.size': 9, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'figure.dpi': 150,
    'savefig.dpi': 600, 'text.usetex': False, 'mathtext.default': 'regular'
})

# --------------------------------------------------
#       Configuration and utility functions
# --------------------------------------------------
CONFIG = {
    'input_file': os.path.join(f"phi1_chi_eBest_{interaction_type}.dat"),
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'figure_size': (8.5, 4.5),
    'x_range': (-230, 130),
    'y_range': (-20, 20),
    'zeta_colors': ['#FF0000', '#008800', '#0000FF', '#FFFFFF', '#888800', '#FF00FF', '#FF0000']
}

def process_data(filename):
    """Load and transform data for plotting"""
    data = pd.read_csv(filename, sep=r'\s+', header=None, names=['phi1', 'chi', 'zeta', 'distance', 'energy'])
    data['phi1_shifted'] = np.where(data['phi1'] < 180, data['phi1'], data['phi1'] - 360)
    data['energy_normalized'] = data['energy'] - 2*CONFIG['energy_ref']
    data['zeta_shifted'] = np.where(data['zeta'] < CONFIG['zeta_max']/2, data['zeta'], data['zeta'] - CONFIG['zeta_max'])
    return data

def create_pivot_tables(data):
    """Create pivot tables for contour plotting"""
    return {
        'energy': data.pivot_table(values='energy_normalized', index='chi', columns='phi1_shifted', aggfunc='mean'),
        'distance': data.pivot_table(values='distance', index='chi', columns='phi1_shifted', aggfunc='mean'),
        'zeta': data.pivot_table(values='zeta_shifted', index='chi', columns='phi1_shifted', aggfunc='mean')
    }

# Load and process data
data = process_data(CONFIG['input_file'])
pivot_tables = create_pivot_tables(data)

print(f"Loaded {len(data)} data points")
print(f"φ₁ range: {data['phi1_shifted'].min():.1f} to {data['phi1_shifted'].max():.1f}")
print(f"χ range: {data['chi'].min():.1f} to {data['chi'].max():.1f}")

pivot_tables['energy'] = pivot_tables['energy'].interpolate(method='linear')
pivot_tables['distance'] = pivot_tables['distance'].interpolate(method='linear')
pivot_tables['zeta'] = pivot_tables['zeta'].interpolate(method='linear')

# Create publication-quality multiplot
def style_axis(ax, hide_xlabel=False):
    """Apply consistent styling to axes"""
    ax.set_xlim(CONFIG['x_range'])
    ax.set_ylim(CONFIG['y_range'])
    ax.set_ylabel(r'$\phi_1$ (°)', fontweight='normal')
    if not hide_xlabel:
        ax.set_xlabel(r'$\chi$ (°)', fontweight='normal')
    ax.set_xticks(np.arange(-200, 121, 40))
    ax.set_yticks(np.arange(-20, 21, 10))
    ax.tick_params(axis='both', which='major', labelsize=9, width=0.8, length=5)
    ax.tick_params(axis='both', which='minor', width=0.6, length=3)
    ax.tick_params(axis='x', labelbottom=not hide_xlabel)
    ax.tick_params(direction='in', which='both')
    ax.tick_params(top=True, right=True, which='both')
    ax.grid(True, linewidth=0.3, color='gray', alpha=0.3, linestyle='-')
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color('#333333')
        spine.set_visible(True)
        
def add_colorbar(im, ax, label, max_ticks=5, decimal_places=2):
    """Add consistent colorbar styling with controlled ticks"""
    cbar = plt.colorbar(im, ax=ax, shrink=1, aspect=8, pad=0.02)
    cbar.set_label(label, rotation=90, labelpad=10, fontsize=9, fontweight='normal')
    
    # Control number of ticks
    cbar.locator = plt.MaxNLocator(nbins=max_ticks)
    cbar.update_ticks()
    
    # Format tick labels to limit decimal places
    cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.{decimal_places}f}'))
    
    cbar.ax.tick_params(labelsize=8, width=0.5, length=3)
    cbar.ax.tick_params(direction='in', which='both')
    cbar.outline.set_linewidth(0.8)
    return cbar

# Create figure and subplots
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=CONFIG['figure_size'])
plt.subplots_adjust(left=0.12, right=0.88, top=0.96, bottom=0.06, hspace=0.1)

# ================================      contourf  ploting      =========================================
# Panel 1: Energy surface
# im1 = ax1.contourf(pivot_tables['energy'].columns, pivot_tables['energy'].index, 
#                      pivot_tables['energy'].values,  levels=50, cmap='viridis',  extend='neither')

# style_axis(ax1, hide_xlabel=True)
# add_colorbar(im1, ax1, 'Binding Energy (eV)', max_ticks=10, decimal_places=2)

# # Panel 2: Distance surface  
# im2 = ax2.contourf(pivot_tables['distance'].columns, pivot_tables['distance'].index,
#                    pivot_tables['distance'].values, levels=50, cmap='plasma', extend='neither')
# style_axis(ax2, hide_xlabel=True)
# add_colorbar(im2, ax2, 'Optimum Distance (Å)', max_ticks=10, decimal_places=1)

# # Panel 3: Zeta surface
# zeta_cmap = LinearSegmentedColormap.from_list('zeta', CONFIG['zeta_colors'])
# im3 = ax3.contourf(pivot_tables['zeta'].columns, pivot_tables['zeta'].index,
#                    pivot_tables['zeta'].values, levels=50, cmap='twilight_shifted', extend='neither')

# # im3 = ax3.pcolormesh(pivot_tables['zeta'].columns, pivot_tables['zeta'].index,
# #                      pivot_tables['zeta'].values, cmap='twilight_shifted', shading='auto')
# style_axis(ax3)
# add_colorbar(im3, ax3, r'$\zeta$ Height Difference (Å)', max_ticks=10, decimal_places=1)

# ================================      imshow  ploting      =========================================
# Panel 1: Energy surface
im1 = ax1.imshow(pivot_tables['energy'].values, cmap='viridis', aspect='auto',
                 extent=[pivot_tables['energy'].columns.min(), pivot_tables['energy'].columns.max(),
                         pivot_tables['energy'].index.min(), pivot_tables['energy'].index.max()],
                 origin='lower', interpolation='bilinear')

style_axis(ax1, hide_xlabel=True)
add_colorbar(im1, ax1, 'Binding Energy (eV)', max_ticks=10, decimal_places=2)

# Panel 2: Distance surface  
im2 = ax2.imshow(pivot_tables['distance'].values, cmap='plasma', aspect='auto',
                 extent=[pivot_tables['distance'].columns.min(), pivot_tables['distance'].columns.max(),
                         pivot_tables['distance'].index.min(), pivot_tables['distance'].index.max()],
                 origin='lower', interpolation='bilinear')
style_axis(ax2, hide_xlabel=True)
add_colorbar(im2, ax2, 'Optimum Distance (Å)', max_ticks=10, decimal_places=1)

# Panel 3: Zeta surface
im3 = ax3.imshow(pivot_tables['zeta'].values, cmap='twilight_shifted', aspect='auto',
                 extent=[pivot_tables['zeta'].columns.min(), pivot_tables['zeta'].columns.max(),
                         pivot_tables['zeta'].index.min(), pivot_tables['zeta'].index.max()],
                 origin='lower', interpolation='bilinear')
style_axis(ax3)
add_colorbar(im3, ax3, r'$\zeta$ Height Difference (Å)', max_ticks=10, decimal_places=1)

# ========================================================================================================

print("Publication-quality multiplot created successfully")
plt.show()

In [ ]:
dat = np.loadtxt(CONFIG['input_file'])

In [ ]:
dat[np.where(dat[:,1] == 10)[0], 2][110:240]

In [ ]:
data['zeta_shifted'].to_numpy()[np.where(dat[:,1] == 10)[0]][110:240]